# 3장 Risk API 확인

## Goal

Compose 또는 Kubernetes에서 실행 중인 동일한 Risk API의 health, model identity와 prediction contract를 확인합니다. 서버 시작과 traffic 실행은 장 README의 명령을 사용합니다. Compose 기본 profile은 baseline이며 Candidate B target runtime을 증명하지 않습니다. 이 Notebook의 static manifest check는 `prepared` evidence이고, local API를 실제로 확인했을 때만 `local_verified` evidence입니다. target deployment success를 증명하지 않습니다.

## Setup

기본 URL은 `http://127.0.0.1:8000`입니다. 다른 URL은 `AIQA_RISK_API_URL` 환경 변수로 지정합니다. API가 실행 중이지 않아도 manifest inspection 부분은 실행됩니다. `API_NOT_RUNNING`은 live runtime evidence가 없다는 blocker이지 model approval을 바꾸는 결과가 아닙니다.

In [1]:
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").is_file() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
assert (ROOT / "pyproject.toml").is_file(), "tta-aiqa repository root를 찾지 못했습니다."
{"repository_root": ROOT.name}


{'repository_root': 'tta-aiqa'}

In [2]:
import os
import requests
import yaml

API_URL = os.getenv("AIQA_RISK_API_URL", "http://127.0.0.1:8000").rstrip("/")
compose = yaml.safe_load((ROOT / "deploy/compose/simple-mlops/compose.yaml").read_text())
assert compose["services"]["risk-api"]["environment"]["AIQA_API_MODEL_BACKEND"] == "local"
API_URL

'http://127.0.0.1:8000'

## Steps

### 1. API 상태와 model identity 조회

In [3]:
def read_json(path: str):
    response = requests.get(f"{API_URL}{path}", timeout=3)
    response.raise_for_status()
    return response.json()

try:
    live_state = {"ready": read_json("/health/ready"), "model": read_json("/v1/model")}
except requests.RequestException as error:
    live_state = {"status": "API_NOT_RUNNING", "next_action": "README의 Compose 실행 명령을 먼저 수행하세요.", "detail": str(error)}
live_state

{'status': 'API_NOT_RUNNING',
 'next_action': 'README의 Compose 실행 명령을 먼저 수행하세요.',
 'detail': 'HTTPConnectionPool(host=\'127.0.0.1\', port=8000): Max retries exceeded with url: /health/ready (Caused by NewConnectionError("HTTPConnection(host=\'127.0.0.1\', port=8000): Failed to establish a new connection: [Errno 61] Connection refused"))'}

### 2. 유효 요청과 contract violation

Operational pool은 target을 포함하지 않는 실제 serving payload입니다. API가 실행
중이면 같은 payload로 정상 200과 intentional contract failure 422를 확인하고,
실행 중이 아니면 다음 action만 표시합니다.

In [4]:
from aiqa_core.adapters.config import load_feature_contract
from traffic_generator.adapters import CsvPatientPool

feature_set = load_feature_contract(ROOT / "configs/contracts/model-input.yaml")
patient_pool = CsvPatientPool(
    ROOT / "data/splits/physionet-2012/revisions/v2/datasets/operational.csv",
    feature_set,
)
valid_features = patient_pool.patient(0)
request_headers = {
    "X-Request-ID": "notebook-valid-request",
    "X-AIQA-Scenario": "baseline",
}

if "model" in live_state:
    valid_response = requests.post(
        f"{API_URL}/v1/predict",
        json={"features": valid_features},
        headers=request_headers,
        timeout=3,
    )
    invalid_features = dict(valid_features)
    invalid_features.pop(next(iter(invalid_features)))
    invalid_response = requests.post(
        f"{API_URL}/v1/predict",
        json={"features": invalid_features},
        headers={"X-Request-ID": "notebook-invalid-request", "X-AIQA-Scenario": "invalid"},
        timeout=3,
    )
    request_exercise = {
        "valid_status": valid_response.status_code,
        "valid_request_id": valid_response.headers.get("X-Request-ID"),
        "invalid_status": invalid_response.status_code,
        "invalid_request_id": invalid_response.headers.get("X-Request-ID"),
        "feature_count": len(valid_features),
    }
    assert request_exercise["valid_status"] == 200
    assert request_exercise["invalid_status"] == 422
else:
    request_exercise = {
        "status": "API_NOT_RUNNING",
        "next_action": "Start the Compose Risk API from the chapter README.",
        "feature_count": len(valid_features),
    }
request_exercise

{'status': 'API_NOT_RUNNING',
 'next_action': 'Start the Compose Risk API from the chapter README.',
 'feature_count': 133}

### 2. Kubernetes model adapter 확인

In [5]:
import json

risk_api_documents = list(
    yaml.safe_load_all((ROOT / "deploy/kubernetes/base/risk-api.yaml").read_text())
)
risk_api_deployment = risk_api_documents[0]
risk_api_environment = {
    item["name"]: item
    for item in risk_api_deployment["spec"]["template"]["spec"]["containers"][0][
        "env"
    ]
}
inference_service = yaml.safe_load(
    (ROOT / "deploy/kubernetes/base/inference-service.yaml").read_text()
)
predictor_environment = {
    item["name"]: item
    for item in inference_service["spec"]["predictor"]["containers"][0]["env"]
}
model_identity = dict(
    line.split("=", maxsplit=1)
    for line in (ROOT / "deploy/kubernetes/base/config/model-identity.env")
    .read_text(encoding="utf-8")
    .splitlines()
    if line and not line.startswith("#")
)
release_manifest = json.loads(
    (ROOT / "docs/reference/evidence/model/revisions/v2/release-manifest.json").read_text(
        encoding="utf-8"
    )
)
risk_api_pull_secrets = risk_api_deployment["spec"]["template"]["spec"][
    "imagePullSecrets"
]
predictor_pull_secrets = inference_service["spec"]["predictor"]["imagePullSecrets"]
expected_digest_reference = predictor_environment["AIQA_KSERVE_EXPECTED_MODEL_SHA256"][
    "valueFrom"
]["configMapKeyRef"]
{
    "backend": risk_api_environment["AIQA_API_MODEL_BACKEND"]["value"],
    "kserve_url": risk_api_environment["AIQA_API_KSERVE_URL"]["value"],
    "expected_model_sha256": model_identity[
        "AIQA_KSERVE_EXPECTED_MODEL_SHA256"
    ],
    "risk_api_image": risk_api_deployment["spec"]["template"]["spec"][
        "containers"
    ][0]["image"],
    "predictor_image": inference_service["spec"]["predictor"]["containers"][0][
        "image"
    ],
    "image_pull_secrets": {
        "risk_api": risk_api_pull_secrets,
        "predictor": predictor_pull_secrets,
    },
    "config_map_reference": expected_digest_reference,
}


{'backend': 'kserve',
 'kserve_url': 'http://mortality-risk-predictor.tta-aiqa.svc.cluster.local',
 'expected_model_sha256': 'f2576f12512a490c9814e5238c3f0d2a421a21637a4b03c882df6ff25a637edc',
 'risk_api_image': 'ghcr.io/seungbaeji/tta-aiqa-risk-api@sha256:922a0a4fa4640fee8ff0e9299ad4b814739f7ade230cc0305cf335fc2f282f65',
 'predictor_image': 'ghcr.io/seungbaeji/tta-aiqa-kserve-predictor@sha256:d7ccc20bc89dca3d37c6768de8986416daf87c626ad828cdbd7889da07811e24',
 'image_pull_secrets': {'risk_api': [{'name': 'ghcr-pull'}],
  'predictor': [{'name': 'ghcr-pull'}]},
 'config_map_reference': {'name': 'model-identity',
  'key': 'AIQA_KSERVE_EXPECTED_MODEL_SHA256'}}

## Checks

In [6]:
assert risk_api_environment["AIQA_API_MODEL_BACKEND"]["value"] == "kserve"
assert "mortality-risk-predictor" in risk_api_environment["AIQA_API_KSERVE_URL"]["value"]
assert risk_api_pull_secrets == [{"name": "ghcr-pull"}]
assert predictor_pull_secrets == [{"name": "ghcr-pull"}]
assert "@sha256:" in risk_api_deployment["spec"]["template"]["spec"]["containers"][0]["image"]
assert "@sha256:" in inference_service["spec"]["predictor"]["containers"][0]["image"]
assert expected_digest_reference == {
    "name": "model-identity",
    "key": "AIQA_KSERVE_EXPECTED_MODEL_SHA256",
}
assert model_identity["AIQA_KSERVE_EXPECTED_MODEL_SHA256"] == release_manifest[
    "model_bundles"
]["baseline/model.joblib"]
if "model" in live_state:
    assert live_state["model"]["profile"] in {"baseline", "candidate-b"}
    assert live_state["ready"]["status"] == "ready"
print("Serving contract checks passed.")


Serving contract checks passed.


## Next Steps

local API가 확인됐으면 Traffic Generator로 baseline 요청을 보낸 뒤 `/metrics`와 Grafana Cloud에서 `model_version`을 확인합니다. target URL, GitOps sync와 live telemetry window는 별도 owner evidence가 필요하므로 없으면 `target_pending`으로 report합니다.